### Libraries And Inputs


In [1]:
import sys
from pathlib import Path

LIB = Path.home() / "Documents/ChatGPT/Nsgm/glwc3/project2/07_Nsgm"
sys.path.insert(0, str(LIB))
import util as yu
import numpy as np
import h5py
import pandas as pd
import pyerrors as pe
import autograd.numpy as anp

files = {"725": LIB / "cB211.072.64/pkl/processData/reg_ignore/cB211.072.64.h5",
         "732": Path.home() / "Desktop/cB211.072.64_732cfg_scalar.h5"}
srijit_file = Path.home() / "Downloads/reproduce_fit_disc_strange_B64_tslow10_tcut2_tmin7.h5"
unit = .018267 * 197.327 / .07948
S = 2
methods = ["JK with Gamma covariance", "Pure JK"]
assert not yu.flag_fast

data = {}
for name, file in files.items():
    tmax = 36 if name == "725" else 31
    with h5py.File(file) as f:
        data[name] = {"c2": f["disc_c2pt"][:, 1:tmax+1].real.astype(float),
                      "vev": f["vev_gSs"][:].real.astype(float),
                      "c3": {t: f[f"disc_gSs/{t}"][:].real.astype(float) for t in range(10, 20)}}
    assert len(data[name]["c2"]) == int(name)

with h5py.File(srijit_file) as f:
    b = f["B64"]
    reference = {"c2": b["raw/c2pt_jk"][:], "vev": b["raw/vev_jk"][:],
                 "c3": {t: b[f"raw/c3pt/ts_{t}_jk"][:] for t in range(10, 20)},
                 "p2": np.column_stack([b[f"twopt/{key}_jk"][:] for key in ("E0", "dE", "c1")]),
                 "points": b["fit/points_jk"][:].T, "covariance": b["fit/covariance"][:],
                 "p3": b["fit/params_jk"][:][[0, 1, 3]].T, "sigma": b["fit/sigma_MeV_jk"][:]}
    ts, ti = b["fit/points_ts"][:], b["fit/points_tins"][:]
    quoted_gamma = (b.attrs["sigma_MeV"], b.attrs["sigma_err_MeV"])
np.testing.assert_array_equal(np.array([ts, ti]).T,
                              [(t, i) for t in range(10, 20) for i in range(2, t//2+1)])


### Raw 725-Configuration Checks


In [2]:
jk = {}
for name, raw in data.items():
    jk[name] = {"c2": yu.jackknife(raw["c2"], d=1),
                "vev": yu.jackknife(raw["vev"], d=1),
                "c3": {t: yu.jackknife(v, d=1) for t, v in raw["c3"].items()}}

pairs = [("C2", data["725"]["c2"], jk["725"]["c2"], reference["c2"]),
         ("VEV", data["725"]["vev"], jk["725"]["vev"], reference["vev"])]
t = 14
pairs.append((f"C3, ts={t}", data["725"]["c3"][t], jk["725"]["c3"][t], reference["c3"][t]))

checks = []
for label, raw, bins, supplied in pairs:
    reproduced = np.concatenate([raw.mean(axis=0)[None, ...], bins])
    np.testing.assert_allclose(reproduced, supplied, rtol=1e-12, atol=1e-20)
    checks.append([label, np.max(np.abs(reproduced-supplied)),
                   np.linalg.norm(reproduced-supplied)/np.linalg.norm(supplied)])
display(pd.DataFrame(checks, columns=["Input", "Max abs difference", "Relative L2 difference"]))


,Input,Max abs difference,Relative L2 difference
0,C2,6.617445e-24,7.702611e-17
1,VEV,1.818989e-12,5.464077e-17
2,"C3, ts=14",5.293956e-22,1.498011e-16


### Two-Point Fits


In [3]:
def model2(p, t):
    return p[0] + anp.log((1+p[2]*anp.exp(-p[1]*t))/(1+p[2]*anp.exp(-p[1]*(t+1))))

obs2, fit2_gamma, p2_jk = {}, {}, {}
for name, raw in data.items():
    # Physical C2 times 8..tmax; the model coordinate is x=t-1.
    t2 = np.arange(7, raw["c2"].shape[1]-1)
    meff_jk = np.log(jk[name]["c2"][:, 7:-1]/jk[name]["c2"][:, 8:])
    obs2[name] = [pe.Obs([v], ["B64"]) for v in raw["c2"].T]
    meff_gamma = [anp.log(a/b) for a, b in zip(obs2[name][7:-1], obs2[name][8:])]
    for value in meff_gamma:
        value.gamma_method(S=S)

    fit2_gamma[name] = pe.fits.least_squares(t2, meff_gamma, model2, correlated_fit=True,
                                             initial_guess=[.379, .205, .67], silent=True)
    fit2_gamma[name].gamma_method(S=S)
    cov_gamma = pe.covariance(meff_gamma)
    cov_jk = yu.jackmec(meff_jk)[2]

    for method in methods:
        mask = cov_gamma/cov_jk if method == "JK with Gamma covariance" else None
        p2_jk[name, method], _, _, warnings = yu.jackfit(
            lambda p: model2(p, t2), meff_jk, [.379, .205, .67], mask=mask,
            ftol=1e-11, xtol=1e-11, gtol=1e-11, maxfev=5000)
        assert warnings == 0 and np.isfinite(p2_jk[name, method]).all()

display(pd.DataFrame({"Srijit": reference["p2"][0],
                      "Reproduced Gamma": [p.value for p in fit2_gamma["725"]]},
                     index=["E0", "dE", "c1"]))


,Srijit,Reproduced Gamma
E0,0.379185,0.379185
dE,0.205035,0.205035
c1,0.668593,0.668593


### Three-Point Ratio Fits (Case IV)


In [4]:
def model3(p):
    return yu.func_ratioSYM_2st_0ra11(ts, ti, *p)

fit3_gamma, points_gamma, covariance_gamma = {}, {}, {}
p3_jk, points_jk, results = {}, {}, {}
for name, raw in data.items():
    c2, vev, c3 = jk[name]["c2"], jk[name]["vev"], jk[name]["c3"]
    ratio_jk = np.column_stack([(c3[t][:, i]+c3[t][:, t-i])/2/c2[:, t-1]-vev
                                for t, i in zip(ts, ti)])
    vev_gamma = pe.Obs([raw["vev"]], ["B64"])
    ratio_gamma = [pe.Obs([(raw["c3"][t][:, i]+raw["c3"][t][:, t-i])/2], ["B64"])
                   /obs2[name][t-1]-vev_gamma for t, i in zip(ts, ti)]
    p2 = fit2_gamma[name]
    # Convert the t-1 overlap coefficient to physical-time r1.
    p2_extra = [p2[1], p2[2]*anp.exp(p2[1])]
    points_gamma[name] = ratio_gamma
    for value in points_gamma[name]:
        value.gamma_method(S=S)
    covariance_gamma[name] = pe.covariance(points_gamma[name])

    def fit_mean(values, **kwargs):
        points = ratio_jk + values[:-2] - ratio_jk.mean(axis=0)
        extra = np.tile(values[-2:], (len(points), 1))
        mask = covariance_gamma[name]/yu.jackmec(points)[2]
        pars, _, _, _ = yu.jackfit(model3, points, [1.13, .145, -.75],
            parsExtra_jk=extra, mask=mask, getFilterInfoQ=True,
            ftol=1e-11, xtol=1e-11, gtol=1e-11, maxfev=5000)
        return pars[0]

    # Keep the external two-point uncertainties and their correlations.
    fit3_gamma[name] = [pe.derived_observable(
        lambda values, **kwargs: fit_mean(values)[j], ratio_gamma + p2_extra,
        num_grad=True, base_step=.1, step_ratio=2) for j in range(3)]
    sigma = fit3_gamma[name][0]*unit
    sigma.gamma_method(S=S)
    results[name] = {"Gamma": (sigma.value, sigma.dvalue)}

    for method in methods:
        p2 = p2_jk[name, method]
        p2_extra = np.column_stack([p2[:, 1], p2[:, 2]*np.exp(p2[:, 1])])
        points = ratio_jk
        points_jk[name, method] = points
        cov_jk = yu.jackmec(points)[2]
        mask = covariance_gamma[name]/cov_jk if method == "JK with Gamma covariance" else None
        p3_jk[name, method], _, _, warnings = yu.jackfit(
            model3, points, [1.13, .145, -.75], mask=mask, parsExtra_jk=p2_extra,
            ftol=1e-11, xtol=1e-11, gtol=1e-11, maxfev=5000)
        assert warnings == 0 and np.isfinite(p3_jk[name, method]).all()
        results[name][method] = yu.jackme(p3_jk[name, method][:, 0]*unit)


### Fit Checks And Comparison


In [5]:
supplied_ratio = np.column_stack([
    (reference["c3"][t][:, i]+reference["c3"][t][:, t-i])/2/reference["c2"][:, t-1]
    - reference["vev"] for t, i in zip(ts, ti)])
difference = points_jk["725", "Pure JK"]-supplied_ratio[1:]
np.testing.assert_allclose(points_jk["725", "Pure JK"], supplied_ratio[1:],
                           rtol=1e-12, atol=2e-11)
print("Original-ratio bins: max difference =", np.max(np.abs(difference)))
display(pd.DataFrame({"Srijit export": reference["p3"][0, [0, 2, 1]],
                      "Direct-ratio Gamma fit": [p.value for p in fit3_gamma["725"]]},
                     index=["g", "Delta E", "r01"]))

print("Srijit's quoted Gamma result:", yu.un2str(*quoted_gamma, precision=3), "MeV")
print("Srijit's supplied fit bins:", yu.jackme_un2str(reference["p3"][1:, 0]*unit, precision=3), "MeV")

table = pd.DataFrame({name: {method: f"{value[0]:.1f}({value[1]:.1f})"
                            for method, value in rows.items()} for name, rows in results.items()})
table.columns = ["725 cfgs [MeV]", "732 cfgs [MeV]"]
display(table)


Original-ratio bins: max difference = 1.0913936421275139e-11
Srijit's quoted Gamma result: 51.4(11.6) MeV
Srijit's supplied fit bins: 51.4(10.4) MeV


,Srijit export,Direct-ratio Gamma fit
g,1.133883,1.117929
Delta E,0.144514,0.155491
r01,-0.750750,-0.742436


,725 cfgs [MeV],732 cfgs [MeV]
Gamma,50.7(10.9),41.4(9.3)
JK with Gamma covariance,50.7(9.6),41.4(8.1)
Pure JK,43.5(7.4),40.6(6.6)
